In [ ]:
import time
import threading
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
# PL is imported in-case you need to clear the Jupyter cache
# if the BD has been modified. sometimes it wont detect
# AXI IP since it cached an old hwh.

from pynq import Overlay, allocate, PL
from pynq.lib import AxiGPIO

%matplotlib inline

BUFFER_SIZE = 48828
# DMA set to transfer 256 beats.
N = 256
# redraw every 0.25 seconds
REDRAW_EVERY_S = 0.25

overlay = Overlay("est.bit")
dma_err = overlay.axi_dma_0
dma_ref = overlay.axi_dma_1
print(list(overlay.ip_dict.keys()))

gpio_block = AxiGPIO(overlay.ip_dict['axi_gpio_0'])
gpio_mux = gpio_block.channel1

mux_state = gpio_mux.read()
gpio_mux.write(1, 0x7)

err_plot = np.zeros(BUFFER_SIZE, dtype=np.int32)
ref_plot = np.zeros(BUFFER_SIZE, dtype=np.int32)
plot_lock = threading.Lock()
stop_event = threading.Event()

def dma():
    ref_a = allocate(shape=(N,), dtype=np.int32)
    ref_b = allocate(shape=(N,), dtype=np.int32)
    err_a = allocate(shape=(N,), dtype=np.int32)
    err_b = allocate(shape=(N,), dtype=np.int32)

    try:
        dma_ref.recvchannel.transfer(ref_a)
        dma_err.recvchannel.transfer(err_a)
        dma_ref.recvchannel.wait()
        dma_err.recvchannel.wait()

        dma_ref.recvchannel.transfer(ref_b)
        dma_err.recvchannel.transfer(err_b)

        while not stop_event.is_set():
            dma_err.recvchannel.wait()
            dma_ref.recvchannel.wait()

            dma_ref.recvchannel.transfer(ref_a)
            dma_err.recvchannel.transfer(err_a)

            with plot_lock:
                err_plot[:-N] = err_plot[N:]
                err_plot[-N:] = err_b
                ref_plot[:-N] = ref_plot[N:]
                ref_plot[-N:] = ref_b

            ref_a, ref_b = ref_b, ref_a
            err_a, err_b = err_b, err_a

    finally:
        ref_a.freebuffer()
        ref_b.freebuffer()
        err_a.freebuffer()
        err_b.freebuffer()


worker = threading.Thread(target=dma, daemon=True)
worker.start()

x = np.arange(BUFFER_SIZE)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
(err_line,) = ax1.plot(x, err_plot, lw=1)
(ref_line,) = ax2.plot(x, ref_plot, lw=1, color="tab:orange")
ax1.set_title("Error")
ax2.set_title("Reference")
ax2.set_xlabel("Sample")
ax1.set_xlim(0, BUFFER_SIZE)
fig.tight_layout()
plt.close(fig)

try:
    while True:
        time.sleep(REDRAW_EVERY_S)

        with plot_lock:
            err_copy = err_plot.copy()
            ref_copy = ref_plot.copy()

        err_line.set_ydata(err_copy)
        ref_line.set_ydata(ref_copy)
        for ax, data in ((ax1, err_copy), (ax2, ref_copy)):
            lo, hi = int(data.min()), int(data.max())
            pad = max(1, (hi - lo) * 0.1)
            ax.set_ylim(-2000000, 2000000)

        clear_output(wait=True)
        display(fig)

except KeyboardInterrupt:
    print("Stopping...")
    stop_event.set()
    worker.join(timeout=1.0)
    from scipy.io import savemat
    with plot_lock:
        savemat("plot_data.mat", {"err_plot": err_plot, "ref_plot": ref_plot})
        err_plot.tofile("err_plot.bin")
        ref_plot.tofile("ref_plot.bin")